In [ ]:
from langgraph.graph import StateGraph, START, END
from IPython.display import Image
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os


In [ ]:
load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')
model_name = os.getenv('GEMINI_MODEL', 'gemini-2.5-flash')

model = ChatGoogleGenerativeAI(
    model=model_name,
    api_key=api_key,
)


In [ ]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str


In [ ]:
def create_outline(state: BlogState) -> dict:
    title = state['title']
    prompt = f'Generate a detailed outline for a blog on the topic: {title}'
    outline = model.invoke(prompt).content
    return {'outline': outline}


In [ ]:
def create_blog(state: BlogState) -> dict:
    title = state['title']
    outline = state['outline']
    prompt = f'Write a detailed blog on the title: {title} using the following outline
{outline}'
    content = model.invoke(prompt).content
    return {'content': content}


In [ ]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()


In [ ]:
initial_state = {'title': 'earth'}
final_state = workflow.invoke(initial_state)

final_state['outline']


In [ ]:
final_state['content']


In [ ]:
Image(workflow.get_graph().draw_mermaid_png())
